# Cell Classification Pipeline with OnClass
 This notebook demonstrates:
 1. Loading pre-trained cell embeddings
 2. Training the OnClass classifier
 3. Evaluating classification performance
 
 Using PBMC dataset and xTrimoGene embeddings as example

In [2]:
import os 
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
import sys
import numpy as np
import pandas as pd
import os
from OnClass.OnClassModel import OnClassModel
from utils import read_ontology_file, read_data, make_folder, read_data_file, read_data, parse_pkl, SplitTrainTest, MapLabel2CL, evaluate, MyDataset, seed_everything
from config import ontology_data_dir, scrna_data_dir, result_dir, optuna_result_dir, cell_emb_dir
from torch.utils.data import DataLoader
import torch
import json

# Set random seeds for reproducibility
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)


Loading data from: /mnt/nvme/extra_data/wujialu/scFM-Bench/data/
Loading cell embeddings from: /mnt/nvme/extra_data/wujialu/scFM-Bench/output


## 1. Configuration
Set paths and parameters

In [3]:
# ======================
# CONFIGURATION 
# ======================

device = "cuda:0"
model = "xTrimoGene"
dnames = "Tabula_Sapiens_all"
batch_col = None
dnames = dnames.split(",")


niter = 5  # 5-fold cross-validation
batch_correct = True
minibatch_size = 128
max_iter = 50
train = True
test_ratio = 0.8

# in-distribution evaluation
dot_product = True 
celltype_embed = "onclass"
refine = False
unseen_ratio_ls = [0]

# Output directory setup
if model is not None:
    output_dir = make_folder(result_dir + f'/{model}/{celltype_embed}_dot_product')
    if model == "xTrimoGene":
        emb_file = f"{model}/mapping_01B-resolution_singlecell_cell_embedding_t4.5_resolution.npy"
    elif model == "scVI" and batch_col is not None:
        emb_file = f"{model}/cell_emb_{batch_col}.npy"
    else:
        emb_file = f"{model}/cell_emb.npy"
else:
    output_dir = make_folder(result_dir + '/Raw')
    emb_file = None

print(f"Device: {device}")
print(f"Model: {model}")
print(f"Datasets: {dnames}")
print(f"Output directory: {output_dir}")

Device: cuda:0
Model: xTrimoGene
Datasets: ['Tabula_Sapiens_all']
Output directory: /home/wujialu/allenwang233/scFM-Bench/OnClass/results_best_params/xTrimoGene/onclass_dot_product


## 2. Load Model Parameters
From optuna optimization results

In [4]:
# Load optimal parameters from optuna results
params_file = optuna_result_dir + "/model_to_params.json"
with open(params_file, "r") as f:
    model_to_params = json.load(f)
    

# For each dataset, extract parameters for the current model
dname_params = {}
for dname in dnames:
    params = model_to_params[dname][model]
    dname_params[dname] = {
        "lr": float(params["lr"]),
        "l2": float(params["l2"])
    }
    print(f"{dname} parameters: lr={dname_params[dname]['lr']}, l2={dname_params[dname]['l2']}")

Tabula_Sapiens_all parameters: lr=0.0001, l2=1e-05


## 3. Main Processing Loop
For a single dataset and single iteration

In [5]:
# === MAIN PROCESSING ===

# Read ontology files (CL ontology)
cell_type_nlp_emb_file, cell_type_network_file, cl_obo_file = read_ontology_file("cl", ontology_data_dir)

# Initialize OnClass model with ontology
OnClass_train_obj = OnClassModel(
    cell_type_nlp_emb_file=cell_type_nlp_emb_file,
    cell_type_network_file=cell_type_network_file,
    device=device
)

# Select first dataset for demonstration
dname = dnames[0]
params = dname_params[dname]
iter = 0  # First iteration only
unseen_ratio = unseen_ratio_ls[0]  # First unseen ratio

print(f"\n{'='*50}")
print(f"Processing dataset: {dname}")
print(f"Iteration: {iter}, Unseen ratio: {unseen_ratio}")
print(f"{'='*50}")

# Create output folder for this run
folder = make_folder(output_dir + '/' + dname + '/' + f"lr_{params['lr']}_l2_{params['l2']}_testset_{test_ratio}" + '/' + str(iter) + '/' + str(unseen_ratio) + '/')
model_path = folder + 'model'
print(f"Results will be saved to: {folder}")

# Load dataset information
data_info_dict = read_data_file(dname, scrna_data_dir)
feature_file = data_info_dict['feature_file']
label_file = data_info_dict['label_file']
gene_file = data_info_dict['gene_file']
filter_key = data_info_dict['filter_key']
label_key = data_info_dict['label_key']
layer_key = data_info_dict['layer_key']
emb_dir = os.path.join(cell_emb_dir, dname, layer_key)

# Read data based on file type
if feature_file.endswith('.pkl'):
    feature, label, genes = parse_pkl(
        feature_file, label_file, gene_file,
        exclude_non_leaf_ontology=True,
        cell_ontology_file=cell_type_network_file
    )
elif feature_file.endswith('.h5ad'):
    feature, genes, label, _, _, remained_terms = read_data(
        feature_file,
        cell_ontology_ids=OnClass_train_obj.cell_ontology_ids,
        exclude_non_leaf_ontology=True,
        tissue_key=None,
        filter_key=filter_key,
        AnnData_label_key=label_key,
        nlp_mapping=False,
        cl_obo_file=cl_obo_file,
        cell_ontology_file=cell_type_network_file,
        co2emb=OnClass_train_obj.co2vec_nlp,
        emb_file=os.path.join(emb_dir, emb_file) if emb_file else None
    )
    # Save remained cell types
    np.save(result_dir + f"/{dname}_remained_celltypes.npy", remained_terms)

print(f"Feature matrix shape: {feature.shape}")
print(f"Labels count: {len(np.unique(label))} cell types")


Processing dataset: Tabula_Sapiens_all
Iteration: 0, Unseen ratio: 0
Results will be saved to: /home/wujialu/allenwang233/scFM-Bench/OnClass/results_best_params/xTrimoGene/onclass_dot_product/Tabula_Sapiens_all/lr_0.0001_l2_1e-05_testset_0.8/0/0/
number of cells before exclude_non_leaf_ontology: 483152


100%|██████████| 2742/2742 [00:00<00:00, 37997.51it/s]


number of cells types being excluded: 40
number of cells after exclude_non_leaf_ontology: 269248
Feature matrix shape: (269248, 3072)
Labels count: 120 cell types


## 4. Train/Test Split & Data Preparation

In [6]:
# Split data into train/test sets
train_feature, train_label, test_feature, test_label, _ = SplitTrainTest(
    feature, label,
    nfold_cls=unseen_ratio,
    random_state=iter,
    nfold_sample=test_ratio
)

print(f"Train size: {len(train_feature)}, Test size: {len(test_feature)}")

# Set genes for train/test
train_genes = genes
test_genes = genes

# Prepare ontology graph if using DAGFormer
if celltype_embed == "DAGFormer":
    OnClass_train_obj.CreateOntoGraph(train_label)

# Embed cell types
OnClass_train_obj.EmbedCellTypes(train_label)
nseen = OnClass_train_obj.nseen
co2i, i2co = OnClass_train_obj.co2i.copy(), OnClass_train_obj.i2co.copy()

# Map labels to ontology indices
train_Y = MapLabel2CL(train_label, co2i)
test_Y = MapLabel2CL(test_label, co2i)


Train size: 53844, Test size: 215404
Shape of Y_emb (2743, 5)


## 5. Feature Processing
With optional batch correction

In [7]:
# Process features (with optional batch correction)
if emb_file is None:
    cor_train_feature, cor_test_feature, cor_train_genes, cor_test_genes = OnClass_train_obj.ProcessTrainFeature(
        train_feature, train_label, train_genes,
        test_feature=test_feature,
        test_genes=test_genes,
        batch_correct=batch_correct,
        log_transform=True
    )
    nhidden = [1000]
else:
    cor_train_feature, cor_test_feature = train_feature, test_feature
    cor_train_genes, cor_test_genes = None, None
    OnClass_train_obj.genes = None
    nhidden = [512, 1024]

print(f"Processed train features: {cor_train_feature.shape}")
print(f"Processed test features: {cor_test_feature.shape}")

# Split train into train/validation
nx = cor_train_feature.shape[0]
ntrain = int(nx * 0.9)
permutation = list(np.random.permutation(nx))
train_ind = permutation[:ntrain]
valid_ind = permutation[ntrain:]

# Create data loaders
train_dataset = MyDataset(cor_train_feature[train_ind, :], train_Y[train_ind])
valid_dataset = MyDataset(cor_train_feature[valid_ind, :], train_Y[valid_ind])
test_dataset = MyDataset(cor_test_feature, test_Y)

train_loader = DataLoader(train_dataset, batch_size=minibatch_size, shuffle=True, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=minibatch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=minibatch_size, shuffle=False, num_workers=4)

Processed train features: (53844, 3072)
Processed test features: (215404, 3072)


## 6. Model Training

In [8]:
# Build and train model
OnClass_train_obj.BuildModel(
    ngene=cor_train_feature.shape[1],
    nhidden=nhidden,
    lr=params["lr"],
    l2=params["l2"],
    dot_product=dot_product
)

# Set feature mean for batch correction
OnClass_train_obj.train_feature_mean = np.mean(cor_train_feature, axis=0)

if train:
    print("\nStarting training...")
    best_valid_loss = float('inf')
    patience = 0
    
    train_losses = []
    valid_losses = []
    
    for epoch in range(max_iter):
        train_epoch_loss, valid_epoch_loss = OnClass_train_obj.Train(train_loader, valid_loader)
        train_losses.append(train_epoch_loss)
        valid_losses.append(valid_epoch_loss)
        
        print(f"Epoch {epoch+1}/{max_iter}: "
              f"Train Loss = {train_epoch_loss:.4f}, "
              f"Valid Loss = {valid_epoch_loss:.4f}")
        
        # Early stopping
        if valid_epoch_loss < best_valid_loss:
            best_valid_loss = valid_epoch_loss
            OnClass_train_obj.save_model(model_path=model_path)
            print(f"Saved model at epoch {epoch+1}")
            patience = 0
        else:
            patience += 1
            if patience >= 5:
                print("Early stopping triggered")
                break



Starting training...


100%|██████████| 43/43 [00:03<00:00, 13.16it/s]


Epoch 1/50: Train Loss = 1.2711, Valid Loss = 0.6799
Saved model at epoch 1


100%|██████████| 43/43 [00:03<00:00, 12.95it/s]


Epoch 2/50: Train Loss = 0.5912, Valid Loss = 0.5396
Saved model at epoch 2


...

Epoch 36/50: Train Loss = 0.1776, Valid Loss = 0.2663


100%|██████████| 43/43 [00:03<00:00, 13.43it/s]


Epoch 37/50: Train Loss = 0.1744, Valid Loss = 0.2720


100%|██████████| 43/43 [00:03<00:00, 13.24it/s]

Epoch 38/50: Train Loss = 0.1720, Valid Loss = 0.2586
Early stopping triggered


## 7. Model Evaluation

In [9]:
# Initialize test model
print(f"\nInitializing test model from {model_path}")
OnClass_test_obj = OnClassModel(
    cell_type_nlp_emb_file=cell_type_nlp_emb_file,
    cell_type_network_file=cell_type_network_file,
    device=device
)

OnClass_test_obj.BuildModel(
    ngene=cor_train_feature.shape[1],
    use_pretrain=model_path,
    dot_product=dot_product
)

# Process test features if needed
if emb_file is None:
    cor_test_feature = OnClass_test_obj.ProcessTestFeature(
        cor_test_feature, 
        cor_test_genes, 
        use_pretrain=model_path,
        batch_correct=batch_correct,
        log_transform=False
    )

# Make predictions
pred_Y_seen, pred_Y_seen_logits, pred_Y_all, pred_label = OnClass_test_obj.Predict(
    test_loader,
    use_normalize=False,
    unseen_ratio=unseen_ratio,
    refine=refine
)

# Convert predictions to cell type names
pred_label = np.array([OnClass_test_obj.i2co[y] for y in pred_label])

# Save predictions
pred_df = pd.DataFrame({
    "y_true": test_label,
    "y_pred": pred_label
})
pred_df.to_csv(folder + "pred_label.csv", index=False)

# Calculate evaluation metrics 
onto_net = OnClass_train_obj.ontology_dict
unseen_l_str = OnClass_train_obj.unseen_co
unseen_l = MapLabel2CL(unseen_l_str, co2i)

res_v = evaluate(
    pred_Y_all, test_Y, unseen_l, nseen,
    Y_net=onto_net, write_screen=True,
    prefix='OnClass', i2co=i2co, train_Y=train_Y
)

# Save metrics
df = pd.DataFrame(res_v.items()).set_index(0).T
df.to_csv(folder + "metrics.csv", index=False)



Initializing test model from /home/wujialu/allenwang233/scFM-Bench/OnClass/results_best_params/xTrimoGene/onclass_dot_product/Tabula_Sapiens_all/lr_0.0001_l2_1e-05_testset_0.8/0/0/model


100%|██████████| 1683/1683 [00:06<00:00, 256.50it/s]


OnClass	0.9984	0.9959	0.9984	0.9959	nan	nan	0.9889	0.9955	0.9149	0.7857	


/home/wujialu/allenwang233/scFM-Bench/OnClass/utils.py:1630: RuntimeWarning: Mean of empty slice
  unseen_auc_macro = np.nanmean(class_auc_macro[unseen_l])
/home/wujialu/allenwang233/scFM-Bench/OnClass/utils.py:1632: RuntimeWarning: Mean of empty slice
  unseen_auprc_macro = np.nanmean(class_auprc_macro[unseen_l])
/home/wujialu/allenwang233/scFM-Bench/OnClass/utils.py:1633: RuntimeWarning: Mean of empty slice
  unseen_f1 = np.nanmean(class_f1[unseen_l])
/home/wujialu/allenwang233/scFM-Bench/OnClass/utils.py:1642: RuntimeWarning: Mean of empty slice
  'Macro F1 >=5000': np.nanmean(class_f1[class_category==4])}
